In [1]:
# key: 

In [2]:
import os

os.environ["WANDB_API_KEY"] = (
    ""
)

In [3]:
# ============================================================
# W&B Results Overview (API-based EDA)
# - pulls runs
# - shows counts, coverage (model x window x target)
# - shows best runs per group
# - quick metric summaries + plots
# ============================================================

import numpy as np
import pandas as pd
import wandb
import matplotlib.pyplot as plt

api = wandb.Api()

# ----------------------------
# 1) Set your project path
# ----------------------------
ENTITY = "julian_oelhaf"  # change if your project is under a team entity
PROJECT = "fc-fl-comparison"  # change if needed
PATH = f"{ENTITY}/{PROJECT}"

print("Loading runs from:", PATH)
runs = api.runs(PATH)
print("Runs iterator ready.")

wandb: Currently logged in as: julian-oelhaf (julian_oelhaf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading runs from: julian_oelhaf/fc-fl-comparison
Runs iterator ready.


In [4]:
# ============================================================
# 2) Pull runs into a flat DataFrame
# ============================================================


def flatten_dict(d, parent="", sep="."):
    out = {}
    d = d or {}
    for k, v in d.items():
        key = f"{parent}{sep}{k}" if parent else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep))
        else:
            out[key] = v
    return out


rows = []
for r in runs:
    cfg = {k: v for k, v in (r.config or {}).items() if not str(k).startswith("_")}
    summ = (r.summary._json_dict if r.summary is not None else {}) or {}

    cfg_f = flatten_dict(cfg)
    summ_f = flatten_dict(summ)

    rows.append(
        {
            "run_id": r.id,
            "name": r.name,
            "state": r.state,
            "created_at": r.created_at,
            "url": r.url,
            **{f"config.{k}": v for k, v in cfg_f.items()},
            **{f"summary.{k}": v for k, v in summ_f.items()},
        }
    )

df = pd.DataFrame(rows)

# Convert likely-numeric columns
for c in df.columns:
    if c.startswith(("config.", "summary.")):
        try:
            df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError):
            pass


print("Total runs:", len(df))
df["state"].value_counts(dropna=False)

df_fin = df[df["state"] == "finished"].copy()
print("Finished runs:", len(df_fin))

Total runs: 596
Finished runs: 574


In [5]:
MODEL_COL = "config.model/name"
WINDOW_COL = "config.window/length_s"
TARGET_COL = "config.task/target_label"

print(MODEL_COL in df.columns, WINDOW_COL in df.columns, TARGET_COL in df.columns)

True True True


In [6]:
# ============================================================
# 3) High-level overview tables
# ============================================================


# Coverage: (model x window x target)
group_cols = [c for c in [MODEL_COL, WINDOW_COL, TARGET_COL] if c is not None]
if group_cols:
    coverage = (
        df.groupby(group_cols)["run_id"]
        .count()
        .rename("n_runs")
        .reset_index()
        .sort_values("n_runs", ascending=False)
    )
    display(coverage.head(50))
else:
    print(
        "Could not detect model/window/target columns. Show columns and set manually."
    )

,config.model/name,config.window/length_s,config.task/target_label,n_runs
55,mlp_regressor,0.02,y_fault_location,59
50,mlp_classifier,0.02,event_type,41
54,mlp_regressor,0.01,y_fault_location,39
58,mlp_regressor,0.05,y_fault_location,36
30,hist_gradient_boosting_regressor,0.02,y_fault_location,33
53,mlp_classifier,0.05,event_type,33
25,hist_gradient_boosting_classifier,0.02,event_type,32
28,hist_gradient_boosting_classifier,0.05,event_type,30
33,hist_gradient_boosting_regressor,0.05,y_fault_location,29
29,hist_gradient_boosting_regressor,0.01,y_fault_location,26


In [7]:
def first_existing(candidates, columns):
    """Return first candidate in columns, else None."""
    for c in candidates:
        if c in columns:
            return c
    return None


def coerce_wandb_metric_series(s: pd.Series) -> pd.Series:
    """
    Converts:
      - numbers
      - strings like '1725298761168 0.9992621141359006 0' -> 0.9992621141359006
      - numeric strings
    to float (NaN if not parseable).
    """

    def parse_one(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return np.nan
        if isinstance(v, (int, float, np.number)):
            return float(v)
        txt = str(v).strip()
        if not txt:
            return np.nan
        parts = txt.split()
        # common W&B "timestamp value step" style
        if len(parts) >= 2:
            try:
                return float(parts[1])
            except Exception:
                pass
        # plain numeric
        try:
            return float(txt)
        except Exception:
            return np.nan

    return s.map(parse_one).astype(float)

In [8]:
# ============================================================
# 5) Pick your primary metric(s)
#    - For classification (FC): accuracy / f1
#    - For regression (FL): mae / rmse
# ============================================================
def first_existing(candidates, columns):
    """Return first candidate in columns, else None."""
    for c in candidates:
        if c in columns:
            return c
    return None


# You can override these manually after you see what's available.
FC_METRIC = first_existing(
    group_cols,
    [
        "summary.cv/mean_f1_score",
        "summary.cv/std_f1_score",
        "summary.cv/mean_precision",
        "summary.cv/std_precision",
        "summary.cv/mean_recall",
        "summary.cv/std_recall",
    ],
)

FL_METRIC = first_existing(
    group_cols,
    [
        "summary.cv/mean_mae",
        "summary.cv/std_mae",
        "summary.cv/mean_rmse",
        "summary.cv/std_rmse",
        "summary.cv/mean_r2",
        "summary.cv/std_r2",
    ]
)

print("FC_METRIC:", FC_METRIC)
print("FL_METRIC:", FL_METRIC)

# Inspect metrics if needed:
# display(pd.Series([c for c in df.columns if c.startswith("summary.cv/")]).to_frame("summary_cols"))

FC_METRIC: None
FL_METRIC: None


In [9]:
cols = df.columns

FC_MEAN = first_existing(
    ["summary.cv.mean_f1_score", "summary.cv/mean_f1_score", "summary.mean_f1_score"],
    cols,
)
FC_STD = first_existing(
    ["summary.cv.std_f1_score", "summary.cv/std_f1_score", "summary.std_f1_score"],
    cols,
)

FL_MEAN_MAE = first_existing(
    ["summary.cv.mean_mae", "summary.cv/mean_mae", "summary.mean_mae"],
    cols,
)
FL_STD_MAE = first_existing(
    ["summary.cv.std_mae", "summary.cv/std_mae", "summary.std_mae"],
    cols,
)

FL_MEAN_R2 = first_existing(
    ["summary.cv.mean_r2", "summary.cv/mean_r2", "summary.mean_r2"],
    cols,
)

print("FC_MEAN:", FC_MEAN, "FC_STD:", FC_STD)
print("FL_MEAN_MAE:", FL_MEAN_MAE, "FL_STD_MAE:", FL_STD_MAE, "FL_MEAN_R2:", FL_MEAN_R2)

FC_MEAN: summary.cv/mean_f1_score FC_STD: summary.cv/std_f1_score
FL_MEAN_MAE: summary.cv/mean_mae FL_STD_MAE: summary.cv/std_mae FL_MEAN_R2: summary.cv/mean_r2


In [10]:
df_fin = df[df["state"] == "finished"].copy()

for m in [FC_MEAN, FC_STD, FL_MEAN_MAE, FL_STD_MAE, FL_MEAN_R2]:
    if m and m in df_fin.columns:
        df_fin[m] = coerce_wandb_metric_series(df_fin[m])

# window as numeric (seconds) — optional, but helps sorting/pivot
if WINDOW_COL in df_fin.columns:
    df_fin[WINDOW_COL] = pd.to_numeric(df_fin[WINDOW_COL], errors="coerce")

In [11]:
df_fc = df_fin[df_fin[TARGET_COL] == "event_type"].copy()
df_fl = df_fin[df_fin[TARGET_COL] == "y_fault_location"].copy()

In [12]:
import numpy as np
import pandas as pd


def export_neurips_fc_main(
    classification_runs: pd.DataFrame,
    *,
    window_col="window_length",
    model_col="model",
    mean_col="mean_f1_score",
    aggfunc="max",
    round_nd=2,
):
    # --------------------------------------------------
    # 1) Pivot: window x model → mean F1
    # --------------------------------------------------
    mean_piv = classification_runs.pivot_table(
        index=window_col,
        columns=model_col,
        values=mean_col,
        aggfunc=aggfunc,
    ).astype(float)


    # --------------------------------------------------
    # 2) Sort windows descending (50 → 10 ms)
    # --------------------------------------------------
    mean_piv = mean_piv.reindex(
        sorted(mean_piv.index, key=lambda x: float(x), reverse=True)
    )

    # --------------------------------------------------
    # 3) Sort models by ASCENDING average mean F1
    # --------------------------------------------------
    model_order = mean_piv.mean(axis=0).sort_values(ascending=True).index
    mean_piv = mean_piv[model_order]

    # --------------------------------------------------
    # 4) NeurIPS layout: models x windows
    # --------------------------------------------------
    table_num = mean_piv.T
    table_num.columns = [f"{int(float(c) * 1000)} ms" for c in table_num.columns]

    # --------------------------------------------------
    # 5) Compute best-per-column mask (numeric only)
    # --------------------------------------------------
    best_mask = table_num.eq(table_num.max(axis=0), axis=1)

    # --------------------------------------------------
    # 6) Format to strings + LaTeX bold
    # --------------------------------------------------
    def fmt(v, is_best):
        if pd.isna(v):
            return "--"
        s = f"{v:.{round_nd}f}"
        return rf"\textbf{{{s}}}" if is_best else s

    table_str = pd.DataFrame(
        {
            col: [fmt(v, best_mask.loc[row, col]) for row, v in table_num[col].items()]
            for col in table_num.columns
        },
        index=table_num.index,
    )

    # --------------------------------------------------
    # 7) Header fix
    # --------------------------------------------------
    table_str.index.name = "Model"

    return table_str


import pandas as pd


def export_neurips_fc_appendix(
    classification_runs: pd.DataFrame,
    *,
    window_col="window_length",
    model_col="model",
    mean_col="mean_f1_score",
    std_col="std_f1_score",
    aggfunc="max",
    mean_nd=3,
    std_nd=3,
    include_std=True,  # appendix: mean ± std (recommended)
    models_sorted_by_mean=True,  # order models by average mean F1
    sort_windows_desc=True,  # 50 → 10 ms
):
    """
    Appendix table for fault classification (F1):
      - rows = horizons (window_length)
      - columns = models ordered by average mean F1 (ascending=False = best on the left)
      - values = mean or mean ± std

    Matches your "Table 3" style.
    """

    df = classification_runs.copy()

    # ---- pivot mean (and std if requested) ----
    mean_piv = df.pivot_table(
        index=window_col,
        columns=model_col,
        values=mean_col,
        aggfunc=aggfunc,
    ).astype(float)

    std_piv = None
    if include_std:
        std_piv = df.pivot_table(
            index=window_col,
            columns=model_col,
            values=std_col,
            aggfunc=aggfunc,
        ).astype(float)

    # ---- sort windows ----
    mean_piv = mean_piv.reindex(
        sorted(mean_piv.index, key=lambda x: float(x), reverse=sort_windows_desc)
    )
    if include_std:
        std_piv = std_piv.reindex(mean_piv.index)

    # ---- sort models by average mean F1 (best first, i.e., descending) ----
    if models_sorted_by_mean:
        model_order = mean_piv.mean(axis=0).sort_values(ascending=False).index
        mean_piv = mean_piv[model_order]
        if include_std:
            std_piv = std_piv[model_order]

    # ---- format cells ----
    if include_std:
        table = (
            mean_piv.map(lambda x: f"{x:.{mean_nd}f}" if pd.notna(x) else "--")
            + r" $\pm$ "
            + std_piv.map(lambda x: f"{x:.{std_nd}f}" if pd.notna(x) else "--")
        )
    else:
        table = mean_piv.map(lambda x: f"{x:.{mean_nd}f}" if pd.notna(x) else "--")

    # ---- pretty index and header ----
    table.index = [f"{int(float(w) * 1000)} ms" for w in table.index]
    table.index.name = "Window"

    return table

In [13]:
df_fc

,run_id,name,state,created_at,url,config.task/type,config.model/name,config.cv/n_splits,config.hgb/max_iter,config.features/flat,...,summary.runtime/predict_single_mean_s/mean,summary.runtime/predict_single_mean_s/std,summary.runtime/predict_single_std_s/mean,summary.runtime/predict_single_std_s/std,summary.runtime/predict_throughput_samples_per_s/mean,summary.runtime/predict_throughput_samples_per_s/std,summary.runtime/scaler_fit_transform_train_s/mean,summary.runtime/scaler_fit_transform_train_s/std,summary.runtime/scaler_transform_test_s/mean,summary.runtime/scaler_transform_test_s/std
133,ka9z0u4k,comic-snowflake-382,finished,2026-01-22T20:57:57Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,3072.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
136,mgf08slv,ethereal-cherry-385,finished,2026-01-22T21:15:30Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,3072.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,gnw81eeo,prime-fire-390,finished,2026-01-22T21:55:49Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,ada_boost_classifier,5.0,NaN,6144.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,ib7m76i9,zesty-pond-391,finished,2026-01-22T21:57:55Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,extra_trees_classifier,5.0,NaN,3072.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,3nghycee,skilled-darkness-392,finished,2026-01-22T22:00:14Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,extra_trees_classifier,5.0,NaN,6144.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588,eeqd7hwj,smooth-night-857,finished,2026-04-20T10:05:37Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,9216.0,...,0.000139,0.000005,0.000027,0.000003,217003.120061,12149.619797,14.519358,0.956895,1.027323,0.015246
589,738tjx9s,still-smoke-858,finished,2026-04-20T10:45:06Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,hist_gradient_boosting_classifier,5.0,100.0,12288.0,...,0.022939,0.005504,0.000731,0.000058,12386.211648,1643.520945,18.742232,1.197670,1.406608,0.027269
592,62fq96jp,eager-microwave-861,finished,2026-04-20T11:11:47Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,hist_gradient_boosting_classifier,5.0,100.0,15360.0,...,0.019940,0.000312,0.000688,0.000132,11988.236721,2061.343483,20.531439,0.499489,1.565497,0.102707
593,58fj4vcq,fast-salad-862,finished,2026-04-20T11:57:14Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,12288.0,...,0.000170,0.000002,0.000034,0.000003,165976.803084,8908.220459,18.456580,0.911734,1.403762,0.020178


In [14]:
# print values of "config.window/length_s"
print(df_fc["config.window/length_s"].unique())

[0.01 0.02 0.03 0.04 0.05]


In [15]:
# ---- FC main table: models x windows, best per window bold ----
table_fc_main = export_neurips_fc_main(
    df_fc.rename(columns={FC_MEAN: "mean_f1_score"}),  # adapt to your function defaults
    window_col=WINDOW_COL,
    model_col=MODEL_COL,
    mean_col="mean_f1_score",
    aggfunc="max",  # take best run if you have multiple seeds
    round_nd=2,
)
display(table_fc_main)

,50 ms,40 ms,30 ms,20 ms,10 ms
Model,,,,,
logistic_regression,0.07,0.08,0.09,0.09,0.10
ada_boost_classifier,--,--,--,0.09,--
ridge_classifier,0.09,0.09,0.09,0.52,0.10
extra_trees_classifier,0.43,0.30,0.21,0.16,0.10
random_forest_classifier,0.84,0.83,0.80,0.77,0.29
k_neighbors_classifier,0.86,0.85,0.83,0.86,0.74
hist_gradient_boosting_classifier,0.99,0.99,0.99,0.97,0.82
mlp_classifier,\textbf{0.99},\textbf{0.99},\textbf{0.99},\textbf{1.00},\textbf{1.00}


In [16]:

# ---- FC appendix: window x models, mean ± std ----
if FC_STD:
    table_fc_app = export_neurips_fc_appendix(
        df_fc.rename(columns={FC_MEAN: "mean_f1_score", FC_STD: "std_f1_score"}),
        window_col=WINDOW_COL,
        model_col=MODEL_COL,
        mean_col="mean_f1_score",
        std_col="std_f1_score",
        aggfunc="max",  # or "mean" if you want to average across seeds here
        mean_nd=3,
        std_nd=3,
        include_std=True,
        models_sorted_by_mean=True,
        sort_windows_desc=True,
    )
    display(table_fc_app)

config.model/name,mlp_classifier,hist_gradient_boosting_classifier,k_neighbors_classifier,random_forest_classifier,extra_trees_classifier,ridge_classifier,ada_boost_classifier,logistic_regression
Window,,,,,,,,
50 ms,0.994 $\pm$ 0.011,0.988 $\pm$ 0.023,0.863 $\pm$ 0.003,0.838 $\pm$ 0.004,0.431 $\pm$ 0.020,0.089 $\pm$ 0.002,-- $\pm$ --,0.071 $\pm$ 0.001
40 ms,0.994 $\pm$ 0.013,0.988 $\pm$ 0.025,0.851 $\pm$ 0.003,0.828 $\pm$ 0.004,0.300 $\pm$ 0.033,0.091 $\pm$ 0.002,-- $\pm$ --,0.079 $\pm$ 0.001
30 ms,0.995 $\pm$ 0.014,0.987 $\pm$ 0.008,0.831 $\pm$ 0.004,0.802 $\pm$ 0.004,0.213 $\pm$ 0.008,0.093 $\pm$ 0.002,-- $\pm$ --,0.086 $\pm$ 0.001
20 ms,0.996 $\pm$ 0.011,0.969 $\pm$ 0.074,0.857 $\pm$ 0.007,0.771 $\pm$ 0.007,0.162 $\pm$ 0.005,0.515 $\pm$ 0.007,0.088 $\pm$ 0.002,0.090 $\pm$ 0.001
10 ms,0.998 $\pm$ 0.019,0.815 $\pm$ 0.024,0.738 $\pm$ 0.012,0.292 $\pm$ 0.026,0.098 $\pm$ 0.001,0.097 $\pm$ 0.002,-- $\pm$ --,0.104 $\pm$ 0.004


In [17]:
# ============================================================
# 6) Best runs per group (model x window x target)
# ============================================================


def best_per_group(df_in, metric, higher_is_better: bool, group_cols):
    d = df_in.dropna(subset=[metric]).copy()
    if d.empty:
        return pd.DataFrame()
    d = d.sort_values(metric, ascending=not higher_is_better)
    # take first row per group after sorting
    best = d.groupby(group_cols, as_index=False).head(1)
    return best


if group_cols and FC_METRIC:
    best_fc = best_per_group(
        df_fin, FC_METRIC, higher_is_better=True, group_cols=group_cols
    )
    cols_show = ["name", "url", "state", "created_at"] + group_cols + [FC_METRIC]
    cols_show = [c for c in cols_show if c in best_fc.columns]
    display(best_fc[cols_show].sort_values(FC_METRIC, ascending=False).head(50))

if group_cols and FL_METRIC:
    # for MAE/RMSE smaller is better
    higher = not any(k in FL_METRIC.lower() for k in ["mae", "rmse", "mse", "loss"])
    best_fl = best_per_group(
        df_fin, FL_METRIC, higher_is_better=higher, group_cols=group_cols
    )
    cols_show = ["name", "url", "state", "created_at"] + group_cols + [FL_METRIC]
    cols_show = [c for c in cols_show if c in best_fl.columns]
    display(best_fl[cols_show].sort_values(FL_METRIC, ascending=True).head(50))

In [18]:
# ============================================================
# 7) Quick summary stats per model/window (median + IQR)
# ============================================================


def summary_stats(df_in, metric, by_cols):
    d = df_in.dropna(subset=[metric]).copy()
    if d.empty:
        return pd.DataFrame()
    g = d.groupby(by_cols)[metric]
    out = g.agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        min="min",
        max="max",
    ).reset_index()
    return out


if MODEL_COL and WINDOW_COL:
    by = [MODEL_COL, WINDOW_COL] + ([TARGET_COL] if TARGET_COL else [])
    if FC_METRIC:
        stats_fc = summary_stats(df_fin, FC_METRIC, by)
        print("FC Summary Stats:")
        display(stats_fc.sort_values(["median"], ascending=False).head(50))
    if FL_METRIC:
        stats_fl = summary_stats(df_fin, FL_METRIC, by)
        print("FL Summary Stats:")
        display(stats_fl.sort_values(["median"], ascending=True).head(50))

In [19]:
# ============================================================
# 8) Simple plots: metric vs window_length per model
#    (boxplot for finished runs)
# ============================================================


def boxplot_metric_vs_window(
    df_in, metric, model_col, window_col, target_col=None, max_models=10
):
    d = df_in.dropna(subset=[metric, model_col, window_col]).copy()
    if d.empty:
        print("No data to plot for:", metric)
        return

    models = sorted(d[model_col].astype(str).unique())[:max_models]
    for m in models:
        dm = d[d[model_col].astype(str) == m].copy()
        dm = dm.sort_values(window_col)

        # optional: choose only one target if many
        title_extra = ""
        if target_col and target_col in dm.columns:
            # if multiple targets exist, plot them separately
            targets = sorted(dm[target_col].astype(str).unique())
            for t in targets:
                dmt = dm[dm[target_col].astype(str) == t]
                if dmt.empty:
                    continue
                labels = [str(x) for x in dmt[window_col].tolist()]
                data = [
                    [v] for v in dmt[metric].tolist()
                ]  # one value per run -> box collapses, still ok
                # Better: group per window
                grouped = dmt.groupby(window_col)[metric].apply(list)
                labels = [str(x) for x in grouped.index.tolist()]
                data = grouped.tolist()

                plt.figure()
                plt.boxplot(data, tick_labels=labels, showfliers=False)
                plt.title(f"{metric} vs window_length | model={m} | target={t}")
                plt.xlabel("window_length")
                plt.ylabel(metric)
                plt.xticks(rotation=45, ha="right")
                plt.tight_layout()
                plt.show()
        else:
            grouped = dm.groupby(window_col)[metric].apply(list)
            labels = [str(x) for x in grouped.index.tolist()]
            data = grouped.tolist()

            plt.figure()
            plt.boxplot(data, tick_labels=labels, showfliers=False)
            plt.title(f"{metric} vs window_length | model={m}")
            plt.xlabel("window_length")
            plt.ylabel(metric)
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()


if FC_METRIC and MODEL_COL and WINDOW_COL:
    boxplot_metric_vs_window(df_fin, FC_METRIC, MODEL_COL, WINDOW_COL, TARGET_COL)

if FL_METRIC and MODEL_COL and WINDOW_COL:
    boxplot_metric_vs_window(df_fin, FL_METRIC, MODEL_COL, WINDOW_COL, TARGET_COL)

In [20]:
# ============================================================
# 9) Export a clean table for paper / debugging
# ============================================================
OUT = "wandb_overview.csv"
df.to_csv(OUT, index=False)
print("Wrote:", OUT)

Wrote: wandb_overview.csv


---

# Ablation studies

In [21]:
ABLATION_PREFIXES = (
    "config.hgb",
    "config.mlp",
)


def is_ablation_param(col: str) -> bool:
    return col.startswith(ABLATION_PREFIXES)

In [22]:
df_fin

,run_id,name,state,created_at,url,config.task/type,config.model/name,config.cv/n_splits,config.hgb/max_iter,config.features/flat,...,summary.runtime/predict_single_mean_s/mean,summary.runtime/predict_single_mean_s/std,summary.runtime/predict_single_std_s/mean,summary.runtime/predict_single_std_s/std,summary.runtime/predict_throughput_samples_per_s/mean,summary.runtime/predict_throughput_samples_per_s/std,summary.runtime/scaler_fit_transform_train_s/mean,summary.runtime/scaler_fit_transform_train_s/std,summary.runtime/scaler_transform_test_s/mean,summary.runtime/scaler_transform_test_s/std
0,3kt9ysi5,snowy-bee-134,finished,2026-01-21T09:43:37Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,50.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9knh61ij,fiery-glade-135,finished,2026-01-21T09:48:44Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ibj0j5wf,vague-haze-137,finished,2026-01-21T09:54:50Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,grxutsz8,efficient-feather-138,finished,2026-01-21T09:58:59Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,300.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,vku5ihg1,sleek-microwave-139,finished,2026-01-21T10:01:52Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,jo4u7jxd,usual-durian-859,finished,2026-04-20T10:56:44Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,0.001792,0.000517,0.000073,0.000053,31277.287367,8542.249739,9.047417,1.265025,0.643260,0.010995
591,gncblfay,bumbling-pyramid-860,finished,2026-04-20T11:05:46Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,mlp_regressor,5.0,NaN,15360.0,...,0.000177,0.000009,0.000044,0.000019,132184.667257,12243.959943,8.601511,1.294458,0.625841,0.033278
592,62fq96jp,eager-microwave-861,finished,2026-04-20T11:11:47Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,hist_gradient_boosting_classifier,5.0,100.0,15360.0,...,0.019940,0.000312,0.000688,0.000132,11988.236721,2061.343483,20.531439,0.499489,1.565497,0.102707
593,58fj4vcq,fast-salad-862,finished,2026-04-20T11:57:14Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,12288.0,...,0.000170,0.000002,0.000034,0.000003,165976.803084,8908.220459,18.456580,0.911734,1.403762,0.020178


In [23]:
rows = []

group_cols = [MODEL_COL, WINDOW_COL]
if TARGET_COL in df_fin.columns:
    group_cols.append(TARGET_COL)

print(group_cols)

['config.model/name', 'config.window/length_s', 'config.task/target_label']


In [24]:
import json
import numpy as np
import pandas as pd


def canonicalize(v):
    """Turn values into hashable, comparable representations."""

    # --- 1) None ---
    if v is None:
        return None

    # --- 2) list / tuple ---
    if isinstance(v, (list, tuple)):
        return json.dumps(list(v), separators=(",", ":"), ensure_ascii=False)

    # --- 3) dict ---
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

    # --- 4) numpy arrays ---
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), separators=(",", ":"), ensure_ascii=False)

    # --- 5) numeric ---
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return float(v)

    # --- 6) strings ---
    if isinstance(v, str):
        s = v.strip()
        try:
            f = float(s)
            return int(f) if f.is_integer() else f
        except Exception:
            return s

    # --- 7) pandas NaN (scalar only!) ---
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    return str(v)


def dedupe_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def sort_mixed(values):
    """Numbers first, then strings (including JSON strings for lists/dicts)."""

    def key(v):
        return (0, float(v)) if isinstance(v, (int, float)) else (1, str(v))

    return sorted(values, key=key)

In [25]:
rows = []

for (model, window, *rest), g in df_fin.groupby(group_cols):
    target = rest[0] if rest else None

    for col in filter(is_ablation_param, df_fin.columns):
        # 1) canonicalize each entry (makes lists hashable)
        canon_vals = [
            v
            for v in (canonicalize(x) for x in g[col].tolist())
            if v is not None and not (isinstance(v, float) and np.isnan(v))
        ]
        # 2) drop Nones + dedupe
        values = dedupe_preserve_order(canon_vals)
        # 3) sort for pretty printing / stable tables
        values = sort_mixed(values)

        # Only treat as ablation if >1 distinct value
        if len(values) <= 1:
            continue

        rows.append(
            {
                "model": model,
                "window_length": window,
                "target": target,
                "ablation_param": col.replace("config.", "").replace("/", "."),
                "n_values": len(values),
                "values": values,
                "n_runs": len(g),
            }
        )

ablations_df = pd.DataFrame(rows)
ablations_df.sort_values(["model", "window_length", "ablation_param"], inplace=True)
ablations_df

,model,window_length,target,ablation_param,n_values,values,n_runs
5,hist_gradient_boosting_classifier,0.01,event_type,hgb.is_depth_limited,2,"[0.0, 1.0]",15
4,hist_gradient_boosting_classifier,0.01,event_type,hgb.l2_regularization,3,"[0.0, 0.0001, 0.01]",15
2,hist_gradient_boosting_classifier,0.01,event_type,hgb.learning_rate,3,"[0.03, 0.1, 0.2]",15
1,hist_gradient_boosting_classifier,0.01,event_type,hgb.max_depth,3,"[3.0, 5.0, 10.0]",15
0,hist_gradient_boosting_classifier,0.01,event_type,hgb.max_iter,3,"[50.0, 100.0, 300.0]",15
...,...,...,...,...,...,...,...
123,mlp_regressor,0.05,y_fault_location,mlp.learning_rate_init,4,"[1e-05, 0.0001, 0.001, 0.01]",35
117,mlp_regressor,0.05,y_fault_location,mlp.max_iter,4,"[100.0, 200.0, 300.0, 400.0]",35
121,mlp_regressor,0.05,y_fault_location,mlp.num_layers,2,"[1.0, 2.0]",35
118,mlp_regressor,0.05,y_fault_location,mlp.width_max,3,"[50.0, 100.0, 256.0]",35


In [26]:
# assumes:
# - df_fin exists
# - columns normalized ("/" -> ".")
# - canonicalize(), dedupe_preserve_order() available


records = []

group_cols = [MODEL_COL, WINDOW_COL]
if TARGET_COL in df_fin.columns:
    group_cols.append(TARGET_COL)

for (model, window, *rest), g in df_fin.groupby(group_cols):
    target = rest[0] if rest else None

    for col in filter(is_ablation_param, df_fin.columns):
        vals = [canonicalize(v) for v in g[col].tolist()]
        vals = [
            v
            for v in vals
            if v is not None and not (isinstance(v, float) and np.isnan(v))
        ]

        if len(set(vals)) <= 1:
            continue

        for v in vals:
            records.append(
                {
                    "model": model,
                    "window_length": window,
                    "target": target,
                    "ablation_param": col.replace("config.", "").replace("/", "."),
                    "ablation_value": v,
                }
            )

runs_per_value = pd.DataFrame(records).value_counts().rename("n_runs").reset_index()

runs_per_value

,model,window_length,target,ablation_param,ablation_value,n_runs
0,mlp_regressor,0.02,y_fault_location,mlp.alpha,0.0001,54
1,mlp_regressor,0.02,y_fault_location,mlp.max_iter,200.0,52
2,mlp_regressor,0.02,y_fault_location,mlp.learning_rate_init,0.001,52
3,mlp_regressor,0.02,y_fault_location,mlp.hidden_layer_sizes,[100],52
4,mlp_regressor,0.02,y_fault_location,mlp.batch_size,auto,50
...,...,...,...,...,...,...
398,hist_gradient_boosting_classifier,0.02,event_type,hgb.max_iter,300.0,1
399,hist_gradient_boosting_classifier,0.02,event_type,hgb.min_samples_leaf,5.0,1
400,mlp_regressor,0.04,y_fault_location,mlp.max_iter,400.0,1
401,mlp_regressor,0.04,y_fault_location,mlp.max_iter,300.0,1


In [27]:
balance_check = (
    runs_per_value.groupby(["model", "window_length", "target", "ablation_param"])
    .agg(
        n_values=("ablation_value", "nunique"),
        run_counts=("n_runs", lambda x: sorted(x.unique().tolist())),
    )
    .reset_index()
)

balance_check["is_unbalanced"] = balance_check["run_counts"].apply(lambda x: len(x) > 1)

unbalanced = balance_check[balance_check["is_unbalanced"]]

display(unbalanced)

print("Total unbalanced ablation settings:", len(unbalanced))

,model,window_length,target,ablation_param,n_values,run_counts,is_unbalanced
0,hist_gradient_boosting_classifier,0.01,event_type,hgb.is_depth_limited,2,"[3, 12]",True
1,hist_gradient_boosting_classifier,0.01,event_type,hgb.l2_regularization,3,"[1, 13]",True
2,hist_gradient_boosting_classifier,0.01,event_type,hgb.learning_rate,3,"[1, 13]",True
4,hist_gradient_boosting_classifier,0.01,event_type,hgb.max_iter,3,"[1, 13]",True
5,hist_gradient_boosting_classifier,0.01,event_type,hgb.min_samples_leaf,3,"[1, 13]",True
...,...,...,...,...,...,...,...
119,mlp_regressor,0.05,y_fault_location,mlp.learning_rate_init,4,"[1, 2, 31]",True
120,mlp_regressor,0.05,y_fault_location,mlp.max_iter,4,"[1, 32]",True
121,mlp_regressor,0.05,y_fault_location,mlp.num_layers,2,"[2, 14]",True
122,mlp_regressor,0.05,y_fault_location,mlp.width_max,3,"[1, 14]",True


Total unbalanced ablation settings: 114


In [28]:
EXPECTED = {
    # -------------------------
    # HistGradientBoosting
    # -------------------------
    "hgb.max_depth": [3, 5, 10],
    "hgb.max_iter": [50, 100, 300],
    "hgb.learning_rate": [0.03, 0.1, 0.2],
    "hgb.min_samples_leaf": [5, 20, 50],
    "hgb.l2_regularization": [0.0, 1e-4, 1e-2],
    # -------------------------
    # MLP
    # -------------------------
    "mlp.hidden_layer_sizes": [
        "[50]",
        "[100]",
        "[100,50]",
        "[256,128]",
    ],
    "mlp.alpha": [1e-5, 1e-4, 1e-3],
    "mlp.learning_rate_init": [1e-5, 1e-4, 1e-3, 1e-2],
    "mlp.batch_size": ["auto", 64, 128, 256],
    "mlp.max_iter": [100, 200, 300, 400],
    "mlp.early_stopping": [False, True],
    "mlp.n_iter_no_change": [5, 10, 20],
}

EXPECTED_RAW = EXPECTED.copy()


def missing_explicit(row):
    exp = set(EXPECTED.get(row["ablation_param"], []))
    if not exp:
        return []
    obs = set(
        runs_per_value[
            (runs_per_value["model"] == row["model"])
            & (runs_per_value["window_length"] == row["window_length"])
            & (runs_per_value["ablation_param"] == row["ablation_param"])
        ]["ablation_value"]
    )
    return sorted(exp - obs)


balance_check["missing_values"] = balance_check.apply(missing_explicit, axis=1)
balance_check["n_missing"] = balance_check["missing_values"].apply(len)

incomplete = balance_check[balance_check["n_missing"] > 0]

incomplete.sort_values("window_length", inplace=True)

display(incomplete)

C:\Users\user\AppData\Local\Temp\ipykernel_31972\2640478497.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  incomplete.sort_values("window_length", inplace=True)


,model,window_length,target,ablation_param,n_values,run_counts,is_unbalanced,missing_values,n_missing


In [29]:
group_cols

['config.model/name', 'config.window/length_s', 'config.task/target_label']

In [30]:
df_fin

,run_id,name,state,created_at,url,config.task/type,config.model/name,config.cv/n_splits,config.hgb/max_iter,config.features/flat,...,summary.runtime/predict_single_mean_s/mean,summary.runtime/predict_single_mean_s/std,summary.runtime/predict_single_std_s/mean,summary.runtime/predict_single_std_s/std,summary.runtime/predict_throughput_samples_per_s/mean,summary.runtime/predict_throughput_samples_per_s/std,summary.runtime/scaler_fit_transform_train_s/mean,summary.runtime/scaler_fit_transform_train_s/std,summary.runtime/scaler_transform_test_s/mean,summary.runtime/scaler_transform_test_s/std
0,3kt9ysi5,snowy-bee-134,finished,2026-01-21T09:43:37Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,50.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9knh61ij,fiery-glade-135,finished,2026-01-21T09:48:44Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ibj0j5wf,vague-haze-137,finished,2026-01-21T09:54:50Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,grxutsz8,efficient-feather-138,finished,2026-01-21T09:58:59Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,300.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,vku5ihg1,sleek-microwave-139,finished,2026-01-21T10:01:52Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,jo4u7jxd,usual-durian-859,finished,2026-04-20T10:56:44Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,0.001792,0.000517,0.000073,0.000053,31277.287367,8542.249739,9.047417,1.265025,0.643260,0.010995
591,gncblfay,bumbling-pyramid-860,finished,2026-04-20T11:05:46Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,mlp_regressor,5.0,NaN,15360.0,...,0.000177,0.000009,0.000044,0.000019,132184.667257,12243.959943,8.601511,1.294458,0.625841,0.033278
592,62fq96jp,eager-microwave-861,finished,2026-04-20T11:11:47Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,hist_gradient_boosting_classifier,5.0,100.0,15360.0,...,0.019940,0.000312,0.000688,0.000132,11988.236721,2061.343483,20.531439,0.499489,1.565497,0.102707
593,58fj4vcq,fast-salad-862,finished,2026-04-20T11:57:14Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,multiclass,mlp_classifier,5.0,NaN,12288.0,...,0.000170,0.000002,0.000034,0.000003,165976.803084,8908.220459,18.456580,0.911734,1.403762,0.020178


In [31]:
# ============================================================
# A) TABLE: Overview of completed ablation studies
#    (complete = all EXPECTED values present AND balanced run counts)
# ============================================================


# ---- EXPECTED must be canonicalized (hashable) ----
# EXPECTED_RAW example:
# EXPECTED_RAW = {"model.mlp.hidden_layer_sizes": [[50],[100],[100,50],[256,128]], ...}
# EXPECTED = {k: [canonicalize(v) for v in vs] for k, vs in EXPECTED_RAW.items()}

# ---- build per-(value) run counts from df_fin ----

records = []
for key, g in df_fin.groupby(group_cols):
    model = key[0]
    window = key[1]
    target = key[2] if len(key) == 3 else None

    for col in filter(is_ablation_param, df_fin.columns):
        # canonicalize series values
        vals = [canonicalize(x) for x in g[col].tolist()]
        vals = [
            v
            for v in vals
            if v is not None and not (isinstance(v, float) and np.isnan(v))
        ]

        # only treat as ablation if more than one distinct value was actually run
        if len(set(vals)) <= 1:
            continue

        ablation_param = col.replace("config.", "").replace("/", ".")
        for v in vals:
            records.append(
                dict(
                    model=model,
                    window_length=window,
                    target=target,
                    ablation_param=ablation_param,
                    ablation_value=v,
                )
            )

runs_per_value = pd.DataFrame(records).value_counts().rename("n_runs").reset_index()


# ---- completeness + balance check (explicit EXPECTED) ----
def missing_explicit(row):
    exp = set(EXPECTED.get(row["ablation_param"], []))
    if not exp:
        return []  # if not specified, treat as "unknown expected"
    obs = set(
        runs_per_value.loc[
            (runs_per_value["model"] == row["model"])
            & (runs_per_value["window_length"] == row["window_length"])
            & (
                runs_per_value["target"].fillna("NA")
                == (row["target"] if row["target"] is not None else "NA")
            )
            & (runs_per_value["ablation_param"] == row["ablation_param"]),
            "ablation_value",
        ].tolist()
    )
    return sorted(exp - obs, key=str)


balance_check = (
    runs_per_value.assign(target=runs_per_value["target"].fillna("NA"))
    .groupby(["model", "window_length", "target", "ablation_param"])
    .agg(
        n_values=("ablation_value", "nunique"),
        run_counts=("n_runs", lambda x: sorted(x.unique().tolist())),
        total_runs=("n_runs", "sum"),
    )
    .reset_index()
)

balance_check["is_unbalanced"] = balance_check["run_counts"].apply(
    lambda xs: len(xs) > 1
)
balance_check["missing_values"] = balance_check.apply(missing_explicit, axis=1)
balance_check["n_missing"] = balance_check["missing_values"].apply(len)

# status
balance_check["status"] = np.select(
    [balance_check["n_missing"] > 0, balance_check["is_unbalanced"]],
    ["❌ incomplete", "⚠️ unbalanced"],
    default="✅ complete",
)

# ---- Overview table: only completed ----
completed_overview = (
    balance_check[balance_check["status"] == "✅ complete"]
    .sort_values(["model", "window_length", "target", "ablation_param"])
    .reset_index(drop=True)
)

display(completed_overview.head(50))
print("Completed ablation studies:", len(completed_overview))

,model,window_length,target,ablation_param,n_values,run_counts,total_runs,is_unbalanced,missing_values,n_missing,status
0,hist_gradient_boosting_classifier,0.01,event_type,hgb.max_depth,3,[1],3,False,[],0,✅ complete
1,hist_gradient_boosting_classifier,0.02,event_type,hgb.max_depth,3,[1],3,False,[],0,✅ complete
2,hist_gradient_boosting_classifier,0.03,event_type,hgb.max_depth,3,[1],3,False,[],0,✅ complete
3,hist_gradient_boosting_classifier,0.04,event_type,hgb.max_depth,3,[1],3,False,[],0,✅ complete
4,hist_gradient_boosting_classifier,0.05,event_type,hgb.max_depth,3,[1],3,False,[],0,✅ complete
5,hist_gradient_boosting_regressor,0.01,y_fault_location,hgb.max_depth,3,[2],6,False,[],0,✅ complete
6,hist_gradient_boosting_regressor,0.02,y_fault_location,hgb.max_depth,3,[1],3,False,[],0,✅ complete
7,hist_gradient_boosting_regressor,0.03,y_fault_location,hgb.max_depth,3,[1],3,False,[],0,✅ complete
8,hist_gradient_boosting_regressor,0.04,y_fault_location,hgb.max_depth,3,[1],3,False,[],0,✅ complete
9,hist_gradient_boosting_regressor,0.05,y_fault_location,hgb.max_depth,3,[1],3,False,[],0,✅ complete


Completed ablation studies: 10


In [32]:
df_fin.head()

,run_id,name,state,created_at,url,config.task/type,config.model/name,config.cv/n_splits,config.hgb/max_iter,config.features/flat,...,summary.runtime/predict_single_mean_s/mean,summary.runtime/predict_single_mean_s/std,summary.runtime/predict_single_std_s/mean,summary.runtime/predict_single_std_s/std,summary.runtime/predict_throughput_samples_per_s/mean,summary.runtime/predict_throughput_samples_per_s/std,summary.runtime/scaler_fit_transform_train_s/mean,summary.runtime/scaler_fit_transform_train_s/std,summary.runtime/scaler_transform_test_s/mean,summary.runtime/scaler_transform_test_s/std
0,3kt9ysi5,snowy-bee-134,finished,2026-01-21T09:43:37Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,50.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9knh61ij,fiery-glade-135,finished,2026-01-21T09:48:44Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ibj0j5wf,vague-haze-137,finished,2026-01-21T09:54:50Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,grxutsz8,efficient-feather-138,finished,2026-01-21T09:58:59Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,300.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,vku5ihg1,sleek-microwave-139,finished,2026-01-21T10:01:52Z,https://wandb.ai/julian_oelhaf/fc-fl-compariso...,regression,hist_gradient_boosting_regressor,5.0,100.0,15360.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# ============================================================
# FIXED VERSION (works with your actual config columns)
# - your df columns are: config.mlp.* and config.hgb.*  (NO "config.model.*")
# - completed_overview ablation_param values are: hgb.max_depth (etc.)
# ============================================================

import random
import numpy as np
import pandas as pd

# ---- metric ----
METRIC_COL = "summary.cv/mean_mae"
HIGHER_IS_BETTER = False
METRIC_COL = METRIC_COL.replace("/", ".")

df_m = df_fin.copy()
df_m.columns = [c.replace("/", ".") for c in df_m.columns]

print("completed_overview rows:", len(completed_overview))
print("metric exists?", METRIC_COL in df_m.columns)
if METRIC_COL in df_m.columns:
    print("metric non-null (finished):", df_m[METRIC_COL].notna().sum())

# ---- baseline defaults: keys MUST match your ablation_param schema (mlp.* / hgb.*) ----
BASE_DEFAULTS_RAW = {
    "hgb.max_depth": None,
    "hgb.max_iter": 100,
    "hgb.learning_rate": 0.1,
    "hgb.min_samples_leaf": 20,
    "hgb.l2_regularization": 0.0,
    "mlp.hidden_layer_sizes": [100],
    "mlp.activation": "relu",
    "mlp.alpha": 1e-4,
    "mlp.learning_rate_init": 1e-3,
    "mlp.batch_size": "auto",
    "mlp.max_iter": 200,
    "mlp.early_stopping": False,
    "mlp.n_iter_no_change": 10,
}
BASE_DEFAULTS = {k: canonicalize(v) for k, v in BASE_DEFAULTS_RAW.items()}


def get_baseline_mean(df_m, model, window, target, metric_col, seed=0):
    mask = (df_m[MODEL_COL.replace("/", ".")] == model) & (df_m[WINDOW_COL.replace("/", ".")] == window)
    if TARGET_COL in df_m.columns:
        mask &= df_m[TARGET_COL.replace("/", ".")] == target

    # baseline for HGB: all HGB params at defaults
    base_mask = mask.copy()
    base_mask &= df_m["config.hgb.max_iter"] == 100
    base_mask &= df_m["config.hgb.learning_rate"] == 0.1
    base_mask &= df_m["config.hgb.min_samples_leaf"] == 20
    base_mask &= df_m["config.hgb.l2_regularization"] == 0.0

    # IMPORTANT: max_depth baseline is "None/null" -> in your df it might be NaN
    # treat NaN as baseline
    base_mask &= df_m["config.hgb.max_depth"].isna()

    base = df_m.loc[base_mask].dropna(subset=[metric_col])
    if base.empty:
        return np.nan

    # if multiple, take mean (or balanced sample if you want)
    return float(base[metric_col].mean())


# ---- mapping: ablation_param -> df column ----
def ablation_param_to_df_col(abp: str) -> str:
    # your df has config.hgb.* / config.mlp.*
    return "config." + abp.replace("/", ".")


def ablation_param_to_base_key(abp: str) -> str:
    return abp.replace("/", ".")


def balanced_value_means(
    df_group: pd.DataFrame, param_col: str, metric_col: str, seed: int = 0
):
    rng = random.Random(seed)
    sub = df_group.dropna(subset=[param_col, metric_col]).copy()
    if sub.empty:
        return {}

    sub["_v"] = [canonicalize(v) for v in sub[param_col].tolist()]
    sub = sub.dropna(subset=["_v"])

    values = sub["_v"].unique().tolist()
    if len(values) <= 1:
        return {}

    idx_by_v = {v: sub.index[sub["_v"] == v].tolist() for v in values}
    k = min(len(idxs) for idxs in idx_by_v.values())
    if k == 0:
        return {}

    means = {}
    for v, idxs in idx_by_v.items():
        pick = idxs if len(idxs) == k else rng.sample(idxs, k)
        means[v] = float(sub.loc[pick, metric_col].mean())
    return means


# ---- compute influence ----
influence_rows = []
skipped = {"param_col_missing": 0, "no_means": 0, "empty_group": 0}

for _, r in completed_overview.iterrows():
    model = r["model"]
    window = r["window_length"]
    target = None if r["target"] == "NA" else r["target"]
    abp = r["ablation_param"]  # e.g. hgb.max_depth / mlp.max_iter

    param_col = ablation_param_to_df_col(abp)
    if param_col not in df_m.columns:
        skipped["param_col_missing"] += 1
        continue

    mask = (df_m[MODEL_COL.replace("/", ".")] == model) & (
        df_m[WINDOW_COL.replace("/", ".")] == window
    )
    if TARGET_COL in df_m.columns:
        if target is None:
            mask &= df_m[TARGET_COL.replace("/", ".")].isna()
        else:
            mask &= df_m[TARGET_COL.replace("/", ".")] == target

    g = df_m.loc[mask].copy()
    if g.empty:
        skipped["empty_group"] += 1
        continue

    means = balanced_value_means(g, param_col=param_col, metric_col=METRIC_COL, seed=0)
    if not means:
        skipped["no_means"] += 1
        continue

    # best/worst depends on metric direction
    best_v = (
        max(means, key=means.get) if HIGHER_IS_BETTER else min(means, key=means.get)
    )
    worst_v = (
        min(means, key=means.get) if HIGHER_IS_BETTER else max(means, key=means.get)
    )

    effect_range = means[best_v] - means[worst_v]

    base_key = ablation_param_to_base_key(abp)
    base_v = BASE_DEFAULTS.get(base_key, None)
    print("Base v for", abp, "is", base_v)
    base_mean = get_baseline_mean(
        df_m, model, window, target, METRIC_COL.replace("/", "."), seed=0
    )
    best_minus_baseline = (
        means[best_v] - base_mean if not np.isnan(base_mean) else np.nan
    )

    influence_rows.append(
        {
            "model": model,
            "window": window,
            "target": target if target is not None else "NA",
            "ablation_param": abp.replace("/", "."),
            "n_values": len(means),
            "best_value": best_v,
            "best_mean": means[best_v],
            "worst_value": worst_v,
            "worst_mean": means[worst_v],
            "effect_range": effect_range,
            "baseline_value": base_v,
            "baseline_mean": base_mean,
            "best_minus_baseline": best_minus_baseline,
        }
    )

influence_df = pd.DataFrame(influence_rows)

print("Built influence rows:", len(influence_df))
print("Skipped:", skipped)

if influence_df.empty:
    print("No influence data to show.")
    display(influence_df)
else:
    print("Top 30 ablation influences by absolute effect range:")
    influence_df["abs_effect"] = influence_df["effect_range"].abs()
    influence_df = influence_df.sort_values("abs_effect", ascending=False).drop(
        columns=["abs_effect"]
    )
    display(influence_df.head(30))

completed_overview rows: 10
metric exists? True
metric non-null (finished): 319
Base v for hgb.max_depth is None
Base v for hgb.max_depth is None
Base v for hgb.max_depth is None
Base v for hgb.max_depth is None
Base v for hgb.max_depth is None
Built influence rows: 5
Skipped: {'param_col_missing': 0, 'no_means': 5, 'empty_group': 0}
Top 30 ablation influences by absolute effect range:


,model,window,target,ablation_param,n_values,best_value,best_mean,worst_value,worst_mean,effect_range,baseline_value,baseline_mean,best_minus_baseline
3,hist_gradient_boosting_regressor,0.04,y_fault_location,hgb.max_depth,3,10.0,14.992571,3.0,20.295373,-5.302802,None,14.590142,0.402429
2,hist_gradient_boosting_regressor,0.03,y_fault_location,hgb.max_depth,3,10.0,15.056422,3.0,20.348189,-5.291767,None,14.641933,0.414489
4,hist_gradient_boosting_regressor,0.05,y_fault_location,hgb.max_depth,3,10.0,15.077206,3.0,20.334032,-5.256826,None,19.937924,-4.860718
1,hist_gradient_boosting_regressor,0.02,y_fault_location,hgb.max_depth,3,10.0,15.226722,3.0,20.311764,-5.085041,None,19.183756,-3.957033
0,hist_gradient_boosting_regressor,0.01,y_fault_location,hgb.max_depth,3,10.0,15.183999,3.0,19.892563,-4.708564,None,14.646097,0.537902


In [34]:
tbl = influence_df.copy()
for c in ["best_mean", "worst_mean", "effect_range", "best_minus_baseline"]:
    if c in tbl.columns:
        tbl[c] = pd.to_numeric(tbl[c], errors="coerce").round(6)

latex = tbl.to_latex(
    index=False,
    escape=False,
    caption=f"Influence of ablation parameters on {METRIC_COL} (balanced means; downsampled per setting).",
    label="tab:ablation_influence",
)

with open("ablation_influence.tex", "w", encoding="utf-8") as f:
    f.write(latex)

print("Wrote ablation_influence.tex")

Wrote ablation_influence.tex
